In [32]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [33]:
df = pd.read_parquet("base_EPEX_df.parquet")
df.head()

,Incentive_forecast,Hour,FORECASTDEMAND,DCOILBRENTEU,sap_lag2d,Price (GBP/MWhe)
0,7589.5,2018-04-17 00:00:00+01:00,24160,73.18,1.7217,41.937710
1,7750.5,2018-04-17 01:00:00+01:00,21620,73.18,1.7217,39.910275
2,7924.0,2018-04-17 02:00:00+01:00,22210,73.18,1.7217,38.426365
3,8143.0,2018-04-17 03:00:00+01:00,22210,73.18,1.7217,31.947200
4,8207.5,2018-04-17 04:00:00+01:00,21000,73.18,1.7217,31.127599


In [34]:
target = "Price (GBP/MWhe)"

# --- Reshape from hourly rows to one row per day ---
d = df.sort_values("Hour").copy()
d["date"] = pd.to_datetime(d["Hour"].dt.date)   # naive local calendar date
d["hod"] = d["Hour"].dt.hour

# Keep only complete 24-hour days (drops DST days with 23/25 hours and partial days)
d = d[d.groupby("date")["Hour"].transform("size") == 24]

def wide(col, prefix):
    w = d.pivot(index="date", columns="hod", values=col)
    w.columns = [f"{prefix}_h{h:02d}" for h in w.columns]
    return w

demand = wide("FORECASTDEMAND", "demand")
wind = wide("Incentive_forecast", "wind")
oil = wide("DCOILBRENTEU", "oil")
gas = wide("sap_lag2d", "gas")
price = wide(target, "price")

# Autoregressive features: shift(freq="D") shifts by true calendar days,
# so missing days produce NaN rather than silently misaligned rows
price_lag1 = price.shift(1, freq="D").add_suffix("_lag1d")   # yesterday's 24 prices
price_lag7 = price.shift(7, freq="D").add_suffix("_lag7d")   # same weekday last week

X = (demand.join(wind, how="inner")
           .join(gas, how="inner")
           .join(oil, how="inner")
           .join(price_lag1, how="inner")
           .join(price_lag7, how="inner"))

X["day_of_week"] = X.index.dayofweek
X["month"] = X.index.month_name
X = X.dropna()
Y = price.loc[X.index]    # 24 columns: the target vector for each day

numeric_features = [c for c in X.columns if c not in ["month", "day_of_week"]]

In [38]:

# --- Pipeline ---
preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), ["day_of_week", "month"]),
])

model = Pipeline([
    ("prep", preprocess),
    ("reg", LinearRegression()),   # multi-output: fits one linear model per hour
])

# --- Time-ordered cross-validation ---
tscv = TimeSeriesSplit(n_splits=5)
rmses = []
for train_idx, test_idx in tscv.split(X):
    model.fit(X.iloc[train_idx], Y.iloc[train_idx])
    pred = model.predict(X.iloc[test_idx])
    rmses.append(np.sqrt(np.mean((Y.iloc[test_idx].values - pred) ** 2)))
print("CV RMSE per fold:", np.round(rmses, 2), "mean:", np.mean(rmses).round(2))

# --- Hold-out evaluation (last 20% of days) ---
split = int(len(X) * 0.8)
model.fit(X.iloc[:split], Y.iloc[:split])
pred = pd.DataFrame(model.predict(X.iloc[split:]), index=Y.index[split:], columns=Y.columns)

err = pred - Y.iloc[split:]
print("Overall MAE :", err.abs().values.mean())
print("Overall RMSE:", np.sqrt((err ** 2).values.mean()))
print("MAE by hour:\n", err.abs().mean().round(2))

CV RMSE per fold: [25.51 69.94 37.58 24.35 18.4 ] mean: 35.16
Overall MAE : 12.482601458131747
Overall RMSE: 18.34189709619441
MAE by hour:
 price_h00     3.87
price_h01     4.68
price_h02     6.44
price_h03     7.66
price_h04     8.66
price_h05    10.30
price_h06    11.34
price_h07    11.58
price_h08    13.30
price_h09    14.10
price_h10    13.93
price_h11    14.39
price_h12    14.85
price_h13    14.77
price_h14    14.51
price_h15    13.65
price_h16    15.79
price_h17    25.23
price_h18    19.94
price_h19    16.59
price_h20    11.69
price_h21     9.53
price_h22    10.04
price_h23    12.75
dtype: float64


In [39]:
def metrics(y_true, y_pred):
    err = np.asarray(y_pred) - np.asarray(y_true)
    return {"MAE": np.abs(err).mean(), "RMSE": np.sqrt((err ** 2).mean())}

Y_test = Y.iloc[split:]
naive_prev_day = X.iloc[split:][[f"{c}_lag1d" for c in Y.columns]].values

results = pd.DataFrame({
    "Regression": metrics(Y_test, pred),
    "Prev day":   metrics(Y_test, naive_prev_day),
}).T
print(results.round(2))

# Per-hour MAE
by_hour = pd.DataFrame({
    "Regression": (pred - Y_test).abs().mean().values,
    "Prev day":   np.abs(naive_prev_day - Y_test.values).mean(axis=0),
}, index=range(24))
print(by_hour.round(2))

# Improvement over baseline (positive = regression is better)
skill = 1 - results.loc["Regression", "RMSE"] / results.loc["Prev day", "RMSE"]
print(f"RMSE improvement vs prev-day baseline: {skill:.1%}")

              MAE   RMSE
Regression  12.48  18.34
Prev day    17.49  28.50
    Regression  Prev day
0         3.87     14.83
1         4.68     14.74
2         6.44     14.89
3         7.66     14.95
4         8.66     15.35
5        10.30     17.02
6        11.34     19.16
7        11.58     21.07
8        13.30     21.55
9        14.10     21.80
10       13.93     21.92
11       14.39     22.26
12       14.85     22.70
13       14.77     22.42
14       14.51     21.64
15       13.65     20.38
16       15.79     17.90
17       25.23     15.84
18       19.94     15.79
19       16.59     14.16
20       11.69     12.06
21        9.53     11.01
22       10.04     11.80
23       12.75     14.60
RMSE improvement vs prev-day baseline: 35.6%
